# 20 — Code Interpreter Agent
## Write Code, Execute It in a Sandbox, Self-Correct on Failure
⏱ ~60 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/20-code-interpreter/code_interpreter_workbook.ipynb)

Most LLMs can write code — but code that *runs correctly on the first try* is a different story.
This workshop builds a **code interpreter agent**: an LLM that writes Python, immediately executes it
in an isolated subprocess, and automatically debugs failures by re-generating code with the error
message as context — looping up to a configurable `MAX_ITERATIONS` limit.

The pattern is the backbone of OpenAI's Code Interpreter plugin, GitHub Copilot's inline runners,
and any agentic system that needs to "think by doing".

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — Code interpreter vs tool-calling; why self-correction matters |
| 2 | **Setup** — Install, API key, imports |
| 3 | **Subprocess Sandbox** — `execute_code()`, timeout, stdout/stderr capture |
| 4 | **State Schema** — `CodeInterpreterState` TypedDict for LangGraph |
| 5 | **Write Code Node** — LLM generates raw Python from a task description |
| 6 | **Retry Prompt** — Feeding back previous code + error for self-correction |
| 7 | **Run Code Node** — Execute and classify success vs failure |
| 8 | **Conditional Router** — `route()` decides loop vs terminate |
| 9 | **Graph Assembly** — Wiring nodes, edges, and conditional branching |
| 10 | **Full Pipeline Run** — Invoke on sample tasks, inspect iterations |
| 11 | **Observability** — Tracing retries, logging intermediate states |
| 12 | **Security Notes** — What subprocess sandboxing does and does NOT protect |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab
- `OPENAI_API_KEY` in `.env` or Colab Secrets
- No external files needed — tasks are defined inline

### Key References
> LangGraph docs: https://langchain-ai.github.io/langgraph/  
> OpenAI Code Interpreter overview: https://platform.openai.com/docs/assistants/tools/code-interpreter  
> Python subprocess docs: https://docs.python.org/3/library/subprocess.html

## Part 1 — Core Concepts

### What is a Code Interpreter Agent?

A **code interpreter agent** closes the loop between *writing* and *running* code:

```
[Natural Language Task]
        |
   LLM writes Python
        |
  subprocess.run()
        |
  success? ──YES──► return output
        |
       NO
        |
  error + previous code ──► LLM (retry)
        |
  (repeat up to MAX_ITERATIONS)
```

### Code Interpreter vs Tool-Calling

| Dimension | Tool-Calling Agent | Code Interpreter Agent |
|-----------|-------------------|-----------------------|
| Output | Structured JSON args | Raw executable code |
| Execution | Pre-built tool function | Arbitrary subprocess |
| Error feedback | Tool raises exception | stderr captured as string |
| Flexibility | Fixed tool surface | Any Python code the LLM can write |
| Security risk | Low (bounded) | Higher (unrestricted exec) |

### Why Self-Correction?

LLMs produce syntactically or logically incorrect code roughly **20-40%** of the time on non-trivial tasks.
Feeding back the *exact* error message — along with the previous code — gives the LLM the context
it needs to pinpoint the bug. Without this loop, you'd need perfect first-attempt generation.

### The Graph Pattern

```
START
  |
write_code     ← LLM writes Python (or fixes prev code + error)
  |
run_code       ← subprocess.run, 10s timeout, capture stdout/stderr
  |
  +── success OR max iterations ────► END
  |
  +── error, iterations < MAX ──────► write_code
```

This is LangGraph's **conditional edge** pattern: `run_code` always leads to a router function
that decides the next node dynamically based on state.

## Part 2 — Setup

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "langchain", "langchain-openai", "langgraph", "python-dotenv"],
        check=True
    )
    print("Colab install complete.")
else:
    print("Local — skipping install (deps expected in venv).")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

In [ ]:
# Core imports used throughout the workshop
import subprocess
import sys
from typing import Literal, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

print("Imports OK")

## Part 3 — Subprocess Sandbox: `execute_code()`

The execution sandbox is the most important safety boundary in this architecture.
We run LLM-generated code with `subprocess.run()`, which:

1. **Isolates the child process** — a crash, `sys.exit()`, or infinite loop in generated code
   cannot bring down the parent Python session.
2. **Enforces a hard timeout** — `timeout=TIMEOUT` raises `subprocess.TimeoutExpired`
   after N seconds; we catch it and return a structured error.
3. **Captures both channels** — `capture_output=True` gives us `stdout` (program output)
   and `stderr` (tracebacks, warnings) as strings, not printed to the terminal.
4. **Returns a structured result** — `returncode == 0` means success; any other value
   (including `-1` for timeout) means failure.

### What does NOT protect you
- File system access: the subprocess can still read/write/delete files.
- Network access: the subprocess can make HTTP requests.
- For production use, run inside a Docker container or gVisor/Firecracker sandbox.

```
execute_code(code: str) -> {
    "stdout":     str,  # captured standard output
    "stderr":     str,  # captured standard error / traceback
    "returncode": int,  # 0 = success, nonzero = failure, -1 = timeout
}
```

In [ ]:
MAX_ITERATIONS = 3
TIMEOUT = 10  # seconds — hard wall; raise for compute-heavy tasks, lower for safety

def execute_code(code: str) -> dict:
    """Run Python code in a subprocess sandbox with a hard timeout."""
    try:
        result = subprocess.run(
            [sys.executable, "-c", code],
            capture_output=True,
            text=True,
            timeout=TIMEOUT,
        )
        return {
            "stdout": result.stdout,
            "stderr": result.stderr,
            "returncode": result.returncode,
        }
    except subprocess.TimeoutExpired:
        return {
            "stdout": "",
            "stderr": f"Execution timed out after {TIMEOUT} seconds.",
            "returncode": -1,
        }

print("execute_code() defined.")

### Smoke-test `execute_code()` with known-good code

In [ ]:
# Test 1: successful execution
good = execute_code("print('hello from sandbox')")
print("returncode:", good["returncode"])  # expect 0
print("stdout:    ", repr(good["stdout"]))
print("stderr:    ", repr(good["stderr"]))

In [ ]:
# Test 2: syntax error — returncode should be nonzero, stderr has the traceback
bad = execute_code("def broken(:\n    pass")
print("returncode:", bad["returncode"])  # expect 1
print("stderr (first 200):", bad["stderr"][:200])

In [ ]:
# Test 3: timeout — set TIMEOUT_TEST=1 so we don't actually wait 10s
import time

def execute_code_with_timeout(code: str, timeout: int) -> dict:
    try:
        result = subprocess.run(
            [sys.executable, "-c", code],
            capture_output=True, text=True, timeout=timeout,
        )
        return {"stdout": result.stdout, "stderr": result.stderr, "returncode": result.returncode}
    except subprocess.TimeoutExpired:
        return {"stdout": "", "stderr": f"Timed out after {timeout}s.", "returncode": -1}

slow = execute_code_with_timeout("import time; time.sleep(5)", timeout=1)
print("returncode:", slow["returncode"])  # expect -1
print("stderr:    ", slow["stderr"])

## Part 4 — State Schema: `CodeInterpreterState`

LangGraph passes a single **state dict** between nodes. We define its shape with a `TypedDict`.
Every node receives the full state and returns a partial dict of keys to update.

| Key | Type | Purpose |
|-----|------|---------|
| `task` | `str` | Natural language task given by the user |
| `code` | `str` | Most recent code generated by the LLM |
| `output` | `str` | stdout from successful execution |
| `error` | `str` | stderr/stdout from failed execution |
| `iterations` | `int` | Number of `write_code` calls so far |

Design notes:
- `TypedDict` provides type hints with zero runtime overhead (unlike dataclasses).
- Keeping `output` and `error` separate lets the router check `state["error"]` as a boolean.
- `iterations` is incremented *inside* `write_code`, not `run_code`, so it accurately counts
  LLM calls — which is the expensive resource to limit.

In [ ]:
class CodeInterpreterState(TypedDict):
    task: str        # natural language coding task
    code: str        # most recent LLM-generated code
    output: str      # stdout from successful run
    error: str       # stderr/stdout from failed run
    iterations: int  # number of write_code calls so far

# Example initial state — this is what you pass to app.invoke()
example_initial_state: CodeInterpreterState = {
    "task": "Print the squares of 1 through 5.",
    "code": "",
    "output": "",
    "error": "",
    "iterations": 0,
}
print("State schema defined:", list(example_initial_state.keys()))

## Part 5 — Write Code Node

The `write_code` node has a single job: **ask the LLM for raw Python code**.

Two important prompt engineering decisions:

1. **`Output ONLY raw Python — no markdown fences, no explanation.`**  
   LLMs love wrapping code in triple-backtick blocks. We want to `exec()` / `subprocess.run()` the
   output directly. The instruction must be explicit and appear at the end of the prompt (recency bias).

2. **Strip markdown defensively**  
   Even with the instruction, some models add fences occasionally. We strip them in code:
   ```python
   code = code.replace("```python", "").replace("```", "").strip()
   ```

The node also increments `iterations` here — so we know how many LLM calls were made.

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini")  # fast and cheap for code gen tasks

def write_code(state: CodeInterpreterState) -> dict:
    """Ask the LLM to write (or fix) Python code for the task."""
    if state["error"]:
        # Retry path: include previous code + error message for self-correction
        prompt = (
            f"Task: {state['task']}\n\n"
            f"Your previous code:\n```python\n{state['code']}\n```\n\n"
            f"Error:\n{state['error']}\n\n"
            "Fix the code. Output ONLY raw Python — no markdown fences, no explanation."
        )
    else:
        # First attempt: clean slate
        prompt = (
            f"Task: {state['task']}\n\n"
            "Write a complete, runnable Python script to solve this task. "
            "Output ONLY raw Python — no markdown fences, no explanation."
        )

    response = llm.invoke([
        SystemMessage(content="You are an expert Python programmer."),
        HumanMessage(content=prompt),
    ])

    # Defensive strip: some models add fences despite the instruction
    code = response.content.strip()
    code = code.replace("```python", "").replace("```", "").strip()

    return {
        "code": code,
        "iterations": state["iterations"] + 1,
    }

print("write_code node defined.")

### Preview: call `write_code` directly on a state dict

You can test any LangGraph node in isolation before wiring it into the graph —
it's just a function that takes state and returns a partial update.

In [ ]:
test_state: CodeInterpreterState = {
    "task": "Print the squares of 1 through 5.",
    "code": "",
    "output": "",
    "error": "",
    "iterations": 0,
}

update = write_code(test_state)
print("iterations after call:", update["iterations"])  # should be 1
print("\nGenerated code:")
print(update["code"])

## Part 6 — Retry Prompt: Self-Correction in Action

The self-correction loop is powered by a **contextual retry prompt**. On any iteration after the
first, the prompt includes:

- The **original task** (so the LLM knows the goal hasn't changed)
- The **previous code** (so the LLM doesn't start from scratch)
- The **exact error message** (so the LLM can pinpoint the bug)

This mimics how a human developer reads a traceback and fixes a bug — not by
rewriting everything, but by understanding what went wrong and making a targeted fix.

Below we simulate a retry scenario manually to see what the LLM does with a broken piece of code:

In [ ]:
# Simulate a state where the previous code had a NameError
broken_state: CodeInterpreterState = {
    "task": "Print the squares of 1 through 5.",
    "code": "for i in range(1, 6):\n    print(i ** squareof)",  # NameError: squareof
    "output": "",
    "error": "NameError: name 'squareof' is not defined",
    "iterations": 1,
}

retry_update = write_code(broken_state)
print("iterations after retry:", retry_update["iterations"])  # should be 2
print("\nFixed code:")
print(retry_update["code"])

## Part 7 — Run Code Node

The `run_code` node calls `execute_code()` and classifies the result:

- `returncode == 0` → **success**: set `output = stdout`, clear `error`
- anything else → **failure**: set `error = stderr or stdout`, clear `output`

Notice we fall back to `stdout` in the error case: some programs write their error messages
to stdout instead of stderr (e.g., certain frameworks and scripts that use `print()` for logging).

The node does NOT decide whether to retry — that's the router's job.

In [ ]:
def run_code(state: CodeInterpreterState) -> dict:
    """Execute the current code and return output or error."""
    result = execute_code(state["code"])
    if result["returncode"] == 0:
        return {"output": result["stdout"], "error": ""}
    # Fall back to stdout if stderr is empty (some scripts print errors to stdout)
    return {"output": "", "error": result["stderr"] or result["stdout"]}

print("run_code node defined.")

In [ ]:
# Test run_code on the code we generated in Part 5
run_state: CodeInterpreterState = {
    "task": "Print squares.",
    "code": update["code"],  # code from the write_code test above
    "output": "",
    "error": "",
    "iterations": 1,
}

run_result = run_code(run_state)
print("output:", repr(run_result["output"]))
print("error: ", repr(run_result["error"]))

## Part 8 — Conditional Router: `route()`

After `run_code`, we need to decide: **retry or terminate?**

LangGraph calls a **router function** after a node when you use `add_conditional_edges()`.
The router inspects state and returns the name of the next node (or `END`).

Our routing logic has exactly two exit conditions:

```
route(state):
    if state["error"] is empty:          → END  (success)
    if state["iterations"] >= MAX:       → END  (gave up)
    otherwise:                           → "write_code"  (retry)
```

The `Literal["write_code", "__end__"]` return type annotation is a LangGraph convention.
It lets the framework validate that your router only returns known node names at build time.

In [ ]:
def route(state: CodeInterpreterState) -> Literal["write_code", "__end__"]:
    """Decide whether to retry (write_code) or terminate (END)."""
    if not state["error"] or state["iterations"] >= MAX_ITERATIONS:
        return END
    return "write_code"

# Unit-test the router
assert route({"error": "",    "iterations": 1}) == END,          "success should end"
assert route({"error": "err", "iterations": 3}) == END,          "max iterations should end"
assert route({"error": "err", "iterations": 1}) == "write_code", "error under limit should retry"
print("All router assertions passed.")

## Part 9 — Graph Assembly

Now we wire the nodes and edges into a compiled LangGraph `StateGraph`.

Key API calls:

| Call | Purpose |
|------|---------|
| `StateGraph(StateClass)` | Create graph with a typed state schema |
| `graph.add_node(name, fn)` | Register a node (name → function) |
| `graph.add_edge(A, B)` | Add an unconditional edge from A to B |
| `graph.add_conditional_edges(A, router)` | After A, call `router(state)` to pick next node |
| `graph.compile()` | Lock topology, return a runnable `CompiledGraph` |

After `compile()`, the graph is immutable. You call `.invoke()`, `.stream()`, or `.batch()` on it.

In [ ]:
def create_workflow():
    """Build and compile the code interpreter graph."""
    graph = StateGraph(CodeInterpreterState)

    # Register nodes
    graph.add_node("write_code", write_code)
    graph.add_node("run_code", run_code)

    # Unconditional edges
    graph.add_edge(START, "write_code")       # entry point
    graph.add_edge("write_code", "run_code")  # always run after writing

    # Conditional edge: after run_code, call route() to decide next step
    graph.add_conditional_edges("run_code", route)

    return graph.compile()

app = create_workflow()
print("Graph compiled:", type(app).__name__)

### Visualize the graph topology

LangGraph can emit a Mermaid diagram of any compiled graph.
Run this to see the nodes and edges drawn out:

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    # Fallback: print the Mermaid source
    print("Graph visualization unavailable:", e)
    print("\nMermaid source:")
    print(app.get_graph().draw_mermaid())

## Part 10 — Full Pipeline Run

With the graph compiled, let's run it on all three sample tasks.

Each invocation starts from a clean initial state. The graph:
1. Calls `write_code` → LLM generates Python
2. Calls `run_code` → subprocess executes it
3. Calls `route` → decides to end or loop
4. If looping, repeats from step 1 with the error context included

In [ ]:
SAMPLE_TASKS = [
    "Write a function that returns the first N Fibonacci numbers and print the first 10.",
    "Sort a list of tuples by the second element in descending order. Use: [(3, 1), (1, 4), (2, 2), (4, 3)]",
    "Write a function to check if a string is a palindrome and test it with 'racecar' and 'hello'.",
]

print(f"Running {len(SAMPLE_TASKS)} tasks...\n")
results = []

for i, task in enumerate(SAMPLE_TASKS, 1):
    print(f"{'=' * 60}")
    print(f"Task {i}: {task}")

    initial_state: CodeInterpreterState = {
        "task": task,
        "code": "",
        "output": "",
        "error": "",
        "iterations": 0,
    }

    result = app.invoke(initial_state)
    results.append(result)

    status = "SUCCESS" if result["output"] else "FAILED"
    print(f"Status: {status}  |  Iterations: {result['iterations']}")
    if result["output"]:
        print(f"Output:\n{result['output'].rstrip()[:300]}")
    if result["error"]:
        print(f"Error:\n{result['error'].rstrip()[:200]}")
    print()

### Inspect the final code for each task

In [ ]:
for i, (task, result) in enumerate(zip(SAMPLE_TASKS, results), 1):
    print(f"--- Task {i}: {task[:50]}... ---")
    print(result["code"])
    print()

## Part 11 — Observability: Streaming Intermediate States

With `app.invoke()`, you only see the final state. For debugging and logging,
use `app.stream()` instead — it yields the state delta after each node.

This is crucial for understanding:
- Which iteration fixed the bug
- What code the LLM generated at each step
- Where latency is concentrated (write vs run)

In [ ]:
stream_task = "Compute the sum of all prime numbers below 50 and print the result."

print(f"Streaming task: {stream_task}\n")

initial: CodeInterpreterState = {
    "task": stream_task,
    "code": "",
    "output": "",
    "error": "",
    "iterations": 0,
}

for step_num, update in enumerate(app.stream(initial), 1):
    node_name, state_update = next(iter(update.items()))
    print(f"Step {step_num} — node: {node_name}")
    for key, val in state_update.items():
        display_val = str(val)[:100] if isinstance(val, str) else val
        print(f"  {key}: {display_val!r}")
    print()

### Iteration counter as a metric

The `iterations` field tells you how many LLM calls were needed — a proxy for task difficulty
and model capability. Track it across many tasks to understand your failure rate:

In [ ]:
print("Summary of completed tasks:")
print(f"{'Task':<60} {'Status':<10} {'Iters'}")
print("-" * 80)
for task, result in zip(SAMPLE_TASKS, results):
    status = "SUCCESS" if result["output"] else "FAILED"
    print(f"{task[:58]:<60} {status:<10} {result['iterations']}")

successes = sum(1 for r in results if r["output"])
total_iters = sum(r["iterations"] for r in results)
print(f"\nSuccess rate: {successes}/{len(results)}  |  Total LLM calls: {total_iters}")

## Part 12 — Configuring Retry Behavior

The two tunable levers are `MAX_ITERATIONS` and `TIMEOUT`.
Their interaction determines the cost/quality trade-off:

| Config | Effect | Use when |
|--------|--------|----------|
| `MAX_ITERATIONS=1` | No retries | Tasks with clear specs, cost-sensitive |
| `MAX_ITERATIONS=3` | Default | General purpose |
| `MAX_ITERATIONS=5` | More retries | Complex/ambiguous tasks |
| `TIMEOUT=5` | Tighter | Simple scripts, safety-first |
| `TIMEOUT=30` | Looser | Data processing, slow IO |

Below we rebuild the workflow with a custom `MAX_ITERATIONS` without touching the node code:

In [ ]:
def create_workflow_with_config(max_iterations: int = 3, timeout: int = 10):
    """Parameterized factory: build the graph with custom retry limits."""

    def _execute_code(code: str) -> dict:
        try:
            r = subprocess.run(
                [sys.executable, "-c", code],
                capture_output=True, text=True, timeout=timeout,
            )
            return {"stdout": r.stdout, "stderr": r.stderr, "returncode": r.returncode}
        except subprocess.TimeoutExpired:
            return {"stdout": "", "stderr": f"Timed out after {timeout}s.", "returncode": -1}

    def _run_code(state: CodeInterpreterState) -> dict:
        result = _execute_code(state["code"])
        if result["returncode"] == 0:
            return {"output": result["stdout"], "error": ""}
        return {"output": "", "error": result["stderr"] or result["stdout"]}

    def _route(state: CodeInterpreterState) -> Literal["write_code", "__end__"]:
        if not state["error"] or state["iterations"] >= max_iterations:
            return END
        return "write_code"

    graph = StateGraph(CodeInterpreterState)
    graph.add_node("write_code", write_code)  # same write_code as before
    graph.add_node("run_code", _run_code)
    graph.add_edge(START, "write_code")
    graph.add_edge("write_code", "run_code")
    graph.add_conditional_edges("run_code", _route)
    return graph.compile()

# Test with max_iterations=1 (no retries)
app_strict = create_workflow_with_config(max_iterations=1, timeout=5)
print("Strict workflow (max_iterations=1) compiled.")

## Part 13 — Security Notes

### What `subprocess.run()` does protect you from

- **Process crash isolation**: if the subprocess segfaults or throws, your parent process is unaffected.
- **`sys.exit()` calls**: a `sys.exit(1)` in generated code only exits the child.
- **Infinite CPU spin**: the `timeout` parameter kills the child after N seconds.
- **Unhandled exceptions**: captured in `stderr`, not propagated to your process.

### What it does NOT protect you from

- **File system access**: LLM-generated code can read/write/delete any file your user can touch.
- **Network access**: the subprocess can make HTTP requests, exfiltrate data, download payloads.
- **Resource exhaustion**: before the timeout fires, the subprocess can use all available RAM.
- **OS-level exploits**: if the LLM generates shellcode or calls `os.system()`, it runs as you.

### Production hardening options

| Technique | Protection level | Complexity |
|-----------|-----------------|------------|
| subprocess only | Low | Simple |
| Docker container | Medium | Moderate |
| gVisor (`runsc`) | High | Moderate |
| Firecracker microVM | Very high | Complex |
| WebAssembly (Pyodide) | High (browser) | Moderate |

For this workshop, the subprocess sandbox is sufficient for educational use with trusted prompts.

In [ ]:
# Demonstrate that a crash in the subprocess does NOT affect this session
crash_result = execute_code("import os; raise RuntimeError('intentional crash')")
print("Parent process survived subprocess crash.")
print("returncode:", crash_result["returncode"])
print("stderr:    ", crash_result["stderr"][:100])

## Part 14 — Extending the Agent

The base pattern is minimal by design. Here are four practical extensions:

### Extension A: Custom task list
Replace `SAMPLE_TASKS` with any list of natural language coding problems.

### Extension B: Structured output
After `run_code`, add a node that parses `output` into structured data (JSON, a number, etc.)
using a second LLM call with `response_format={"type": "json_object"}`.

### Extension C: Multi-language support
Replace `sys.executable` with `node`, `ruby`, `Rscript`, etc. in `execute_code()`.
The LLM prompt changes to match the target language.

### Extension D: Test-driven generation
Add a test harness step after `run_code`: run the generated code against a set of
expected inputs/outputs, and treat test failures as errors to feed back for correction.

Let's try Extension A — a harder task requiring iteration:

In [ ]:
harder_task = (
    "Write a Python script that: "
    "(1) generates a list of 20 random integers between 1 and 100, "
    "(2) computes the mean, median, and mode, "
    "(3) prints each statistic on its own line with its label."
)

harder_result = app.invoke({
    "task": harder_task,
    "code": "",
    "output": "",
    "error": "",
    "iterations": 0,
})

print(f"Iterations: {harder_result['iterations']}")
print(f"Status: {'SUCCESS' if harder_result['output'] else 'FAILED'}")
if harder_result["output"]:
    print(f"Output:\n{harder_result['output'].rstrip()}")

## Part 15 — Complete `create_workflow()` from Source

For reference, here is the full production implementation from `src/workflow.py`,
identical to what ships in the example:

In [ ]:
# Full create_workflow() — matches src/workflow.py exactly

def create_workflow_production():
    """Production version matching src/workflow.py."""
    llm = ChatOpenAI(model="gpt-4o-mini")

    def write_code(state: CodeInterpreterState) -> dict:
        if state["error"]:
            prompt = (
                f"Task: {state['task']}\n\n"
                f"Your previous code:\n```python\n{state['code']}\n```\n\n"
                f"Error:\n{state['error']}\n\n"
                "Fix the code. Output ONLY raw Python — no markdown fences, no explanation."
            )
        else:
            prompt = (
                f"Task: {state['task']}\n\n"
                "Write a complete, runnable Python script to solve this task. "
                "Output ONLY raw Python — no markdown fences, no explanation."
            )
        response = llm.invoke([
            SystemMessage(content="You are an expert Python programmer."),
            HumanMessage(content=prompt),
        ])
        return {"code": response.content.strip(), "iterations": state["iterations"] + 1}

    def run_code(state: CodeInterpreterState) -> dict:
        result = execute_code(state["code"])
        if result["returncode"] == 0:
            return {"output": result["stdout"], "error": ""}
        return {"output": "", "error": result["stderr"] or result["stdout"]}

    def route(state: CodeInterpreterState) -> Literal["write_code", "__end__"]:
        if not state["error"] or state["iterations"] >= MAX_ITERATIONS:
            return END
        return "write_code"

    graph = StateGraph(CodeInterpreterState)
    graph.add_node("write_code", write_code)
    graph.add_node("run_code", run_code)
    graph.add_edge(START, "write_code")
    graph.add_edge("write_code", "run_code")
    graph.add_conditional_edges("run_code", route)
    return graph.compile()

app_prod = create_workflow_production()
print("Production workflow compiled:", type(app_prod).__name__)

## Part 16 — Run the Production `main()` Logic

The `main.py` entry point wraps the workflow in a loop over `SAMPLE_TASKS`
and prints a formatted report. Here is the equivalent inline:

In [ ]:
# Mirrors main.py exactly
app_final = create_workflow()

for task in SAMPLE_TASKS:
    print(f"\n{'=' * 60}")
    print(f"Task: {task}")
    result = app_final.invoke({
        "task": task,
        "code": "",
        "output": "",
        "error": "",
        "iterations": 0,
    })
    status = "SUCCESS" if result["output"] else "FAILED"
    print(f"Status: {status}  |  Iterations: {result['iterations']}")
    if result["output"]:
        print(f"\nOutput:\n{result['output'].rstrip()}")
    if result["error"]:
        print(f"\nError:\n{result['error'].rstrip()}")
    print(f"\nFinal code:\n{result['code']}")

## Part 17 — Key Concepts Recap

Before the exercises, let's consolidate what we've covered:

### The Self-Correction Loop (write → run → route)

```
state["iterations"] = 0, state["error"] = ""
        |
   write_code: LLM generates Python
        |       iterations += 1
   run_code:  subprocess.run(code, timeout=10)
        |
   route():
     ├─ no error → END (success)
     ├─ iterations >= MAX → END (gave up)
     └─ error, under limit → write_code (retry with error context)
```

### Why this works
1. **Subprocess isolation** — crashes in generated code don't crash the agent.
2. **Error context in retry prompt** — the LLM sees exactly what went wrong.
3. **MAX_ITERATIONS guard** — prevents infinite (and expensive) retry spirals.
4. **Conditional edges** — LangGraph's routing is type-safe and easy to test in isolation.

### State evolution across iterations

| Iteration | task | code | output | error | iterations |
|-----------|------|------|--------|-------|------------|
| Initial   | set  | ""   | ""     | ""    | 0 |
| After write_code #1 | set | new code | "" | "" | 1 |
| After run_code (fail) | set | same | "" | traceback | 1 |
| After write_code #2 | set | fixed code | "" | "" | 2 |
| After run_code (success) | set | same | output | "" | 2 |

---
## Exercises

Work through these exercises to solidify your understanding.
Try each one before checking the answer key below.

---

### Exercise 1 — Timeout Enforcement

Modify `execute_code()` to use `TIMEOUT = 2` seconds, then invoke the agent on this task:

```python
task = "Write a Python script that sleeps for 5 seconds then prints 'done'."
```

Expected behavior: the agent times out, treats it as an error, and retries with the timeout
message as context. Observe how many iterations it takes (probably all 3, then gives up).

---

### Exercise 2 — Deliberate First-Attempt Failure

Add a 4th task to `SAMPLE_TASKS` that will almost certainly fail on first attempt:

```python
"Import the library 'nonexistent_lib' and print its version."
```

Run the agent and observe how it handles a `ModuleNotFoundError`. What does the LLM do
on retry? Does it recover?

---

### Exercise 3 — Expose the Prompt

Add a `verbose=True` parameter to `write_code`. When `verbose=True`, print the full
prompt that is sent to the LLM before each API call. This is invaluable for debugging
prompt engineering issues in production.

---
## Answer Key

### Exercise 1 — Answer

In [ ]:
# Exercise 1 — Timeout enforcement with TIMEOUT=2

TIMEOUT_EX1 = 2  # deliberately short

def execute_code_ex1(code: str) -> dict:
    """execute_code with a 2-second timeout."""
    try:
        result = subprocess.run(
            [sys.executable, "-c", code],
            capture_output=True,
            text=True,
            timeout=TIMEOUT_EX1,
        )
        return {"stdout": result.stdout, "stderr": result.stderr, "returncode": result.returncode}
    except subprocess.TimeoutExpired:
        return {
            "stdout": "",
            "stderr": f"Execution timed out after {TIMEOUT_EX1} seconds.",
            "returncode": -1,
        }

def run_code_ex1(state: CodeInterpreterState) -> dict:
    result = execute_code_ex1(state["code"])
    if result["returncode"] == 0:
        return {"output": result["stdout"], "error": ""}
    return {"output": "", "error": result["stderr"] or result["stdout"]}

# Build a workflow using the short-timeout run_code
graph_ex1 = StateGraph(CodeInterpreterState)
graph_ex1.add_node("write_code", write_code)  # same LLM node
graph_ex1.add_node("run_code", run_code_ex1)  # short-timeout execution
graph_ex1.add_edge(START, "write_code")
graph_ex1.add_edge("write_code", "run_code")
graph_ex1.add_conditional_edges("run_code", route)
app_ex1 = graph_ex1.compile()

timeout_task = "Write a Python script that sleeps for 5 seconds then prints 'done'."
result_ex1 = app_ex1.invoke({
    "task": timeout_task, "code": "", "output": "", "error": "", "iterations": 0
})

print(f"Iterations: {result_ex1['iterations']}")
print(f"Status: {'SUCCESS' if result_ex1['output'] else 'FAILED (as expected)'}")
print(f"Final error: {result_ex1['error'][:200]}")
print("\n[Expected: 3 iterations, all timed out, final state is FAILED]")

### Exercise 2 — Answer

In [ ]:
# Exercise 2 — Deliberate ModuleNotFoundError on first attempt

impossible_task = "Import the library 'nonexistent_lib' and print its version."

result_ex2 = app.invoke({
    "task": impossible_task,
    "code": "",
    "output": "",
    "error": "",
    "iterations": 0,
})

print(f"Iterations: {result_ex2['iterations']}")
print(f"Status: {'SUCCESS' if result_ex2['output'] else 'FAILED'}")
print()
print("Final code:")
print(result_ex2["code"])
if result_ex2["error"]:
    print("\nFinal error:")
    print(result_ex2["error"][:300])

print()
print("[Observation: the LLM likely tries pip install, then alternative approaches, then gives up]")

### Exercise 3 — Answer

In [ ]:
# Exercise 3 — Verbose mode: print the full prompt before each LLM call

def write_code_verbose(state: CodeInterpreterState, verbose: bool = True) -> dict:
    """write_code with optional prompt logging."""
    if state["error"]:
        prompt = (
            f"Task: {state['task']}\n\n"
            f"Your previous code:\n```python\n{state['code']}\n```\n\n"
            f"Error:\n{state['error']}\n\n"
            "Fix the code. Output ONLY raw Python — no markdown fences, no explanation."
        )
    else:
        prompt = (
            f"Task: {state['task']}\n\n"
            "Write a complete, runnable Python script to solve this task. "
            "Output ONLY raw Python — no markdown fences, no explanation."
        )

    if verbose:
        print(f"\n[PROMPT — iteration {state['iterations'] + 1}]")
        print("-" * 40)
        print(prompt)
        print("-" * 40)

    response = llm.invoke([
        SystemMessage(content="You are an expert Python programmer."),
        HumanMessage(content=prompt),
    ])
    code = response.content.strip().replace("```python", "").replace("```", "").strip()
    return {"code": code, "iterations": state["iterations"] + 1}

# Wire the verbose node into a test graph
graph_ex3 = StateGraph(CodeInterpreterState)
graph_ex3.add_node("write_code", write_code_verbose)
graph_ex3.add_node("run_code", run_code)
graph_ex3.add_edge(START, "write_code")
graph_ex3.add_edge("write_code", "run_code")
graph_ex3.add_conditional_edges("run_code", route)
app_ex3 = graph_ex3.compile()

verbose_result = app_ex3.invoke({
    "task": "Print the first 5 powers of 2.",
    "code": "", "output": "", "error": "", "iterations": 0,
})
print("\nFinal output:", verbose_result["output"].strip())

---
## Workshop Complete

You've built a **code interpreter agent** from scratch:

- `execute_code()` — subprocess sandbox with hard timeout
- `CodeInterpreterState` — typed state schema for LangGraph
- `write_code` node — LLM generates or repairs Python
- `run_code` node — executes code, classifies success/failure
- `route()` — conditional edge deciding retry vs terminate
- Full graph: `START → write_code → run_code → [loop | END]`

**Next: Example 21 — Multi-Agent Supervisor**  
Learn how to orchestrate multiple specialized agents under a single supervisor that routes
tasks to the right sub-agent based on the query type.